# M03.3 — Train detector D0 (ModernBERT, PERSON token classification)

Run top-to-bottom on **Google Colab** (T4 GPU, `Runtime -> Run all`). Expect **~45-90 min**:
the dataset (~130 MB) and ModernBERT (~600 MB) come from the Drive cache on re-runs.

What this produces — the frozen pilot detector **D0**:

- trained on **train_side** rows only (POSITIVES: rows with PERSON spans; NEGATIVES: the
  ~71% of rows without one — kept in for the precision guardrail),
- labels built with the repo's verified alignment seam (`bio_labels_from_spans`) using
  ModernBERT's own tokenizer (mBERT arrays never reused),
- model selection on the **dev_calibration** file only — the final-eval partition is
  never loaded beyond its lock ids,
- saved to Drive: best checkpoint (by dev span-F1), metrics, per-epoch curve, error
  examples, and a manifest with versions/seed/config,
- acceptance signals printed as PASS/FAIL: nontrivial dev span-F1, working precision on
  negatives, and resource usage recorded.

The output is a candidate D0. Freezing it for RL is an owner decision after reviewing
the dev metrics and error examples.

In [ ]:
import datetime
import json
import os
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
RUN_ID = datetime.datetime.now(datetime.UTC).strftime("d0-%Y%m%d-%H%M%S")
WORK = os.path.join(DRIVE_ROOT, RUN_ID)
CACHE = os.path.join(DRIVE_ROOT, "cache")
EXPECTED_FINAL_EVAL_SHA = "513335badf5752a340698d3c7efa7029a8780c51fde2675c80ef4208ce27d39c"

SEED = 20260920          # project seed; all sampling here is seeded
MAX_LENGTH = 512         # token truncation; truncated-span rows are counted and reported
BATCH_SIZE = 16
EPOCHS = 3              # ~1h wall-clock on T4; best epoch by dev span-F1 is kept
LR = 5e-5

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

REPORT = {"run_id": RUN_ID, "checks": {}, "config": {
    "seed": SEED, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
    "epochs": EPOCHS, "lr": LR,
}}


def check(name, ok, detail=""):
    REPORT["checks"][name] = {"ok": bool(ok), "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))


print(f"run_id: {RUN_ID}  ->  {WORK}")

In [ ]:
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True
)

import numpy as np  # noqa: E402
import torch  # noqa: E402
import transformers  # noqa: E402

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from pii_redteam.data import QuarantinedRecord, to_canonical  # noqa: E402
from pii_redteam.detector import (  # noqa: E402
    bio_labels_from_spans,
    spans_from_bio_labels,
    trim_span_whitespace,
)

torch.manual_seed(SEED)
np.random.seed(SEED)

REPORT["environment"] = {
    "repo_commit": REPO_COMMIT,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
}
print(json.dumps(REPORT["environment"], indent=2))
assert torch.cuda.is_available(), "This notebook needs a GPU runtime."

In [ ]:
import shutil

# Rebuild the locked splits from the pinned revision and assert the lock.
splits_dir = os.path.join(REPO_DIR, "artifacts", "splits")
subprocess.run(
    [sys.executable, "scripts/carve_splits.py", "--cache-dir", CACHE, "--out", splits_dir],
    cwd=REPO_DIR, env=dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src")), check=True,
)
manifest = json.load(open(os.path.join(splits_dir, "final_eval_manifest.json")))
check(
    "final-eval lock reproduces",
    manifest["final_eval_sha256"] == EXPECTED_FINAL_EVAL_SHA,
    manifest["final_eval_sha256"],
)
if manifest["final_eval_sha256"] != EXPECTED_FINAL_EVAL_SHA:
    raise SystemExit("Lock mismatch - refusing to train.")
FINAL_EVAL_IDS = {r["example_id"] for r in manifest["rows"]}


def load_rows(path):
    rows, quarantined = [], 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                canonical = to_canonical(json.loads(line))
            except QuarantinedRecord:
                quarantined += 1
                continue
            if canonical["example_id"] not in FINAL_EVAL_IDS:
                rows.append(canonical)
    return rows, quarantined


train_rows, q1 = load_rows(os.path.join(CACHE, "1english_openpii_30k.jsonl"))
dev_rows, q2 = load_rows(os.path.join(CACHE, "1english_openpii_8k.jsonl"))
train_pos = [r for r in train_rows if r["spans"]]
dev_pos = [r for r in dev_rows if r["spans"]]
REPORT["data"] = {
    "train_rows": len(train_rows), "train_quarantined": q1, "train_pos": len(train_pos),
    "dev_rows": len(dev_rows), "dev_quarantined": q2, "dev_pos": len(dev_pos),
}
print(json.dumps(REPORT["data"], indent=2))

In [ ]:
import time

from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer

MB_ID = "answerdotai/ModernBERT-base"
LABEL2ID = {"O": 0, "B-PERSON": 1, "I-PERSON": 2}
tok = AutoTokenizer.from_pretrained(MB_ID)
model = AutoModelForTokenClassification.from_pretrained(
    MB_ID, num_labels=3,
    id2label={0: "O", 1: "B-PERSON", 2: "I-PERSON"}, label2id=LABEL2ID,
).cuda()
model.gradient_checkpointing_disable()


class SpanDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
        self.truncated_spans_lost = 0

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        enc = tok(row["text"], return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
        offsets = enc["offset_mapping"]
        labels = bio_labels_from_spans(offsets, row["spans"])
        ids = []
        for (s, e), lab in zip(offsets, labels, strict=True):
            ids.append(-100 if (s, e) == (0, 0) else LABEL2ID[lab])
        # spans that no longer fit after truncation are lost for this row: count once
        visible_end = offsets[-2][1] if len(offsets) > 2 else 0
        lost = [sp for sp in row["spans"] if sp["end"] > visible_end]
        if lost:
            self.truncated_spans_lost += len(lost)
        return {
            "input_ids": enc["input_ids"], "labels": ids,
            "offsets": offsets, "text": row["text"],
            "gold": [(sp["start"], sp["end"]) for sp in row["spans"]],
            "example_id": row["example_id"],
        }


def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id or 0
    input_ids, attn, labels = [], [], []
    for b in batch:
        n = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad] * n)
        attn.append([1] * len(b["input_ids"]) + [0] * n)
        labels.append(b["labels"] + [-100] * n)
    return {
        "input_ids": torch.tensor(input_ids), "attention_mask": torch.tensor(attn),
        "labels": torch.tensor(labels),
    }


def spans_from_probs(logits, offsets, text):
    pred = logits.argmax(-1).tolist()
    labels = [model.config.id2label[i] for i in pred]
    decoded = spans_from_bio_labels(offsets, labels)
    return [(sp["start"], sp["end"]) for sp in trim_span_whitespace(text, decoded)]


@torch.no_grad()
def evaluate(model, rows, batch_size=32):
    model.eval()
    tp = fp = fn = 0
    partial = 0
    neg_rows = neg_fp = 0
    errors = []
    for i in range(0, len(rows), batch_size):
        # tokenize on the fly for eval (keeps memory flat)
        enc = tok([r["text"] for r in rows[i:i + batch_size]],
                  return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH,
                  padding=True)
        input_ids = torch.tensor(enc["input_ids"]).cuda()
        attn = torch.tensor(enc["attention_mask"]).cuda()
        logits = model(input_ids=input_ids, attention_mask=attn).logits
        for j, row in enumerate(rows[i:i + batch_size]):
            offsets = enc["offset_mapping"][j]
            pred = spans_from_probs(logits[j], offsets, row["text"])
            gold = [(sp["start"], sp["end"]) for sp in row["spans"]]
            pred_set, gold_set = set(pred), set(gold)
            tp += len(pred_set & gold_set)
            fp += len(pred_set - gold_set)
            fn += len(gold_set - pred_set)
            if not gold:
                neg_rows += 1
                neg_fp += len(pred_set)
            for p in pred_set - gold_set:
                for g in gold_set:
                    if p[0] < g[1] and p[1] > g[0]:
                        partial += 1
                        break
            if (pred_set != gold_set) and len(errors) < 20:
                errors.append({
                    "example_id": row["example_id"], "text": row["text"][:300],
                    "gold": sorted(gold_set), "pred": sorted(pred_set),
                })
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "span_precision": round(precision, 4), "span_recall": round(recall, 4),
        "span_f1": round(f1, 4), "tp": tp, "fp": fp, "fn": fn,
        "partial_overlap_preds": partial, "neg_rows": neg_rows, "neg_false_positives": neg_fp,
        "errors": errors,
    }


train_ds = SpanDataset(train_rows)
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=collate, num_workers=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
best = {"f1": -1.0, "epoch": -1}
curve = []
torch.cuda.reset_peak_memory_stats()
t_start = time.time()
steps_per_sec = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].cuda()
        attn = batch["attention_mask"].cuda()
        labels = batch["labels"].cuda()
        with torch.autocast("cuda", dtype=scaler_dtype):
            loss = model(input_ids=input_ids, attention_mask=attn, labels=labels).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == 0:
            print(f"epoch {epoch} first-step loss: {loss.item():.4f}")
    train_s = time.time() - t0
    print(f"epoch {epoch}: train {train_s:.0f}s (truncated spans lost so far: {train_ds.truncated_spans_lost})")
    metrics = evaluate(model, dev_rows)
    curve.append({"epoch": epoch, "train_seconds": round(train_s, 1),
                  "train_loss": round(loss.item(), 4), **{k: metrics[k] for k in
                  ("span_precision", "span_recall", "span_f1", "fp", "fn", "neg_false_positives")}})
    print(f"epoch {epoch} dev: {json.dumps({k: metrics[k] for k in ('span_precision','span_recall','span_f1')})}")
    if metrics["span_f1"] > best["f1"]:
        best = {"f1": metrics["span_f1"], "epoch": epoch}
        model.save_pretrained(os.path.join(WORK, "d0-best"))
        tok.save_pretrained(os.path.join(WORK, "d0-best"))
total_s = time.time() - t_start
peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2)
steps = len(loader) * EPOCHS
REPORT["training"] = {
    "model_id": MB_ID, "steps": steps, "total_seconds": round(total_s, 1),
    "steps_per_second": round(steps / total_s, 2), "peak_mem_gb": peak_gb,
    "truncated_spans_lost": train_ds.truncated_spans_lost,
    "best_epoch": best["epoch"], "best_dev_f1": best["f1"], "curve": curve,
}
print(json.dumps(REPORT["training"], indent=2))

In [ ]:
# Acceptance signals (plan M03: "useful nontrivial performance, not labeling every
# capitalized token"). These are pilot floors for the D0 CANDIDATE, not final-eval claims.
final_metrics = evaluate(model, dev_rows)
check("nontrivial dev span-F1 (>= 0.70)", final_metrics["span_f1"] >= 0.70,
      f"dev span_f1={final_metrics['span_f1']}")
check("precision holds on negatives", final_metrics["neg_false_positives"] <= 0.02 * max(final_metrics["neg_rows"], 1),
      f"{final_metrics['neg_false_positives']} FPs on {final_metrics['neg_rows']} negative dev rows")
check("best checkpoint saved", os.path.exists(os.path.join(WORK, "d0-best", "config.json")),
      f"epoch {REPORT['training']['best_epoch']}")

REPORT["dev_metrics"] = {k: v for k, v in final_metrics.items() if k != "errors"}
REPORT["errors"] = final_metrics["errors"]
REPORT["finished_utc"] = datetime.datetime.now(datetime.UTC).isoformat()

with open(os.path.join(WORK, "d0_metrics.json"), "w") as f:
    json.dump(REPORT, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("D0 TRAINING SUMMARY (dev_calibration only; final_eval untouched)")
print("=" * 70)
for name, res in REPORT["checks"].items():
    print(f"[{'PASS' if res['ok'] else 'FAIL'}] {name} -- {res['detail']}")
print(json.dumps(REPORT["dev_metrics"], indent=2))
print(f"checkpoint: {WORK}/d0-best")
print(f"metrics:    {WORK}/d0_metrics.json")
print("Next: review error examples; if acceptable, freeze this checkpoint as D0 in STATE.md.")
if not all(r["ok"] for r in REPORT["checks"].values()):
    print("NOTE: acceptance floors not met - inspect errors before any RL work.")